In [1]:
import ROOT

import numpy as np
import matplotlib.pyplot as plt
import scipy.interpolate as interp
import scipy.optimize as opt
import matplotlib as mpl
import scipy.special as sf
import scipy.integrate as integ
import scipy.linalg as la
mpl.rc('xtick', direction='in', top=True)
mpl.rc('ytick', direction='in', right=True)
mpl.rc('xtick.minor', visible=True)
mpl.rc('ytick.minor', visible=True)
import os
import re
import uuid
import json
import copy
from tabulate import tabulate




# Load

In [2]:
base_path1="Band_class_training data/TE_011/Results"
index_path1 = os.path.join(base_path1, "Te011.json")
with open(index_path1, "r") as f:
    TE011_Index = json.load(f)

base_path2 = "Band_class_training data/TE_012/Results"
index_path2 = os.path.join(base_path2, "Te012.json")
with open(index_path2, "r") as f:
    TE012_Index = json.load(f)

In [3]:
base_path1="testing_data/TE011results"
index_path1 = os.path.join(base_path1, "Te011.json")
with open(index_path1, "r") as f:
    TE011Testing_Index = json.load(f)

base_path2="testing_data/TE012results"
index_path2 = os.path.join(base_path2, "Te012.json")
with open(index_path2, "r") as f:
    TE012Testing_Index = json.load(f)

# Functions

In [4]:
def more_feature_inputs_outputs(index1,index2,spread=9,noise=False,SNR=1,verbose=False,minpeaks=True,freq_diff_features=False,widetol=False):
    """
    This Function takes 2 premade Jason indexes containing a file structure of the angle position and then histograms of a simulation data ROOT file
    The method to create these jason files can be found in data_load.ipynb within this same directory.

    Inputs

    Index1: the TE011 Jason index
    Index2: The TE012 Jason index

    spread=9 : this is used in several points it gneraly refers to the number of frequency bins arround a central point that can be considerd within the same peak or is likly to be that same peak
    Noise=False: When true adds noise to the PSDs based on an SNR from the highest peak
    SNR=1: said SNR
    verbose=False: if true prints an update on progress as the fuction runs
    minpeaks=True: When true will output 5 relvent peaks corisponding to the 5 posible sidebands when false will output 10 relivent peaks 5 from TE011 5 from TE012
    Freq_diff_features=False: adds features to the output of every frequenceas difference to every other frequency

    Outputs

    all_sidebands: a nx5x2 array of the location and power of each sideband in the order -2,0,2,-1,1

    Inputs_sorted: an array of the classifier inputs sorted by frequency npsds x nPeaks x nfeatures

    Outputs_sorted: an array of the classifier outputs npsd x nPeaks with labels -2,-1,0,1,2,np.inf corisponding to the input peak at the same locatin if it is a sideband with said lable or junk/noise np.inf


    """

    #Spread needs to be odd
    if spread % 2 != 1:
        spread += 1

    # arrays of the "locations" of sidebands
    band1 = np.array([-2, 0, 2,])
    band2 = np.array([-1,1])

    #preparing arrays
    all_sidebands = []
    all_maxbands = []
    all_outputs=[]

    ang_pos=[]


    jsn_name = "LocustEventProperties.json"

    #Begin iteration through angles the angles for Index1 and Index2 should be the same
    for angle in index1:

        #Begin iteration through position the angles for Index1 and Index2 should be the same
        for position in index1[angle]:

            #Progress printing
            if verbose:
                print(f"Sidebands:Processing angle={angle}, position={position}", flush=True)

            #loading relevent information from the event properties jason file
            jsn_info = index1[angle][position][jsn_name]
            jsn_filepath = jsn_info["filepath"]

            with open(jsn_filepath, "r") as f:
                jsn = json.load(f)
        
            avgax = jsn["0"]["0"]["avg-axial-frequency"]
            central = jsn["0"]["0"]["output-avg-frequency"]

            #Loading and storing all of the filepaths for each PSD
            psd_by_filepath1 = {}
            psd_by_filepath2 = {}

            for hist_name, hist_info in index1[angle][position].items():
                if "PSD" not in hist_name:
                    continue

                filepath = hist_info["filepath"]

                if filepath not in psd_by_filepath1:
                    psd_by_filepath1[filepath] = []

                psd_by_filepath1[filepath].append(hist_name)  
            
            for hist_name, hist_info in index2[angle][position].items():
                if "PSD" not in hist_name:
                    continue

                filepath = hist_info["filepath"]

                if filepath not in psd_by_filepath2:
                    psd_by_filepath2[filepath] = []

                psd_by_filepath2[filepath].append(hist_name)

            # Begin Itteration through each file

            for (filepath1, hist_names), (filepath2, _) in zip(psd_by_filepath1.items(), psd_by_filepath2.items()):
                file1 = ROOT.TFile.Open(filepath1)
                file2 = ROOT.TFile.Open(filepath2)
                if not file1 or file1.IsZombie():
                    raise RuntimeError(f"Could not open ROOT file: {filepath}")
                if not file2 or file2.IsZombie():
                    raise RuntimeError(f"Could not open ROOT file: {filepath}")
                try:
                    # Begin Itteration through each Hist

                    for hist_name in hist_names:
                        hist1 = file1.Get(hist_name)
                        hist2 = file2.Get(hist_name)
                        if not hist1 or not hist2:
                            raise RuntimeError(
                                f"Missing histogram {hist_name} in file {filepath}"
                            )
                        nbins = hist1.GetNbinsX()
                        
                        bins = range(1, nbins + 1)

                        #Loading powers and frequencies from both Hists
                        power1 = []
                        power2 = []
                        center1 = []
                        center2 = []

                        for i in bins:
                            power1.append(hist1.GetBinContent(i))
                            power2.append(hist2.GetBinContent(i))
                            center = hist1.GetBinCenter(i)
                            center1.append(center)
                            center2.append(center)

                        power1 = np.array(power1)
                        power2 = np.array(power2)
                        center1 = np.array(center1)
                        center2 = np.array(center2)

                        space1=np.mean(np.diff(center1))
                        space2=np.mean(np.diff(center2))
                        
                        # Finding and storing the Location and Power of each sideband -2,0,2 from TE011 and -1,1 from TE012

                        sidebands=[]

                        for band in band1:

                            target = central + (avgax * band)
                            mask = np.abs(center1 - target) <= (space1 * spread)

                            if np.any(mask):

                                local_idx = np.argmax(power1[mask])
                                max_idx = np.where(mask)[0][local_idx]
                                power = power1[max_idx]

                                if power == 0:
                                    loc = np.inf
                                    pos=0
                                else:
                                    loc = int(band)
                                    pos=center1[max_idx]

                                result = (power,pos, loc)
                            else:
                                result = (0, 0, np.inf)
                                
                            sidebands.append(result)

                        for band in band2:

                            target = central + (avgax * band)
                            mask = np.abs(center2 - target) <= (space2 * spread)

                            if np.any(mask):
 
                                local_idx = np.argmax(power2[mask])
                                max_idx = np.where(mask)[0][local_idx]
                                power = power2[max_idx]

                                if power == 0:
                                    loc = np.inf
                                    pos=0
                                else:
                                    loc = int(band)
                                    pos=center2[max_idx]

                                result = (power,pos, loc)
                            else:
                                result = (0, 0, np.inf)

                            sidebands.append(result)
                            
                            
                        sidebands = np.array(sidebands)


                        #Finding and loading all relevent maximum peaks for inputs

                        maxbands1=[]
                        maxbands2=[]
                        power_sim1 = power1.copy()
                        power_sim2 = power2.copy()

                        #Adding Noise  
                        if noise:
                            mx1=np.max(power_sim1)
                            mx2=np.max(power_sim2)
                            sigma1=np.sqrt(mx1/(2*SNR))
                            sigma2=np.sqrt(mx2/(2*SNR))
                            noise_power1 = np.abs(np.random.normal(0, sigma1, nbins) + 1j*np.random.normal(0, sigma1, nbins))**2
                            noise_power2 = np.abs(np.random.normal(0, sigma2, nbins) + 1j*np.random.normal(0, sigma2, nbins))**2
                            
                            power_sim1+=noise_power1
                            power_sim2+=noise_power2

                        max_peaks_found1=0 
                        max_peaks_found2=0 

                        if minpeaks:
                            maxpeaks1=3
                            maxpeaks2=2
                        else:
                            maxpeaks1=5
                            maxpeaks2=5

                        while max_peaks_found1<maxpeaks1:

                            mx=np.argmax(power_sim1)
                            center=float(center1[mx]) 
                            power=float(power_sim1[mx])

                            if len(maxbands1) > 0:
                                matches = np.where(
                                    np.abs(np.array(maxbands1)[:,1] - center) <= space1 * spread)[0]
                            else:
                                matches = []

                            if len(matches) == 1 and power !=0 :
                                maxbands1[matches[0]][0]+=power
                            else:
                                maxbands1.append([power,center])
                                max_peaks_found1+=1
                            
                            power_sim1[mx] = 0
                            
                        while max_peaks_found2<maxpeaks2:

                            mx=np.argmax(power_sim2)
                            center=float(center2[mx]) 
                            power=float(power_sim2[mx])

                            if len(maxbands2) > 0:
                                matches = np.where(
                                    np.abs(np.array(maxbands2)[:,1] - center) <= space2 * spread)[0]
                            else:
                                matches = []

                            if len(matches) == 1 and power !=0 :
                                maxbands2[matches[0]][0]+=power
                            else:
                                maxbands2.append([power,center])
                                max_peaks_found2+=1

                            power_sim2[mx] = 0

                        maxbands1=np.array(maxbands1)
                        maxbands2=np.array(maxbands2)
                        
                        maxbands1_sorted = maxbands1[np.argsort(maxbands1[:, 1])]
                        maxbands2_sorted = maxbands2[np.argsort(maxbands2[:, 1])]

                        maxbands = np.vstack((maxbands1_sorted, maxbands2_sorted))

                        # Creating outputs by coparing sidebands and maxbands to find matching peaks

                        ar1=maxbands
                        ar2=sidebands
                        if widetol:
                            toler=space1*(spread/2)
                        else:
                            toler= 1e-6

                        match_matrix = np.isclose(ar1[:, None, 1],ar2[None, :, 1],rtol=0,atol=toler)
                        ar1_i, ar2_i = np.where(match_matrix)
                        out = np.full(ar1.shape[0], np.inf)
                        out[ar1_i] = ar2[ar2_i, 2]
                        if minpeaks:
                        
                            out[:3][np.isin(out[:3], [-1, 1])] = np.inf
                            out[3:][np.isin(out[3:], [-2, 0, 2])] = np.inf
                        else:
                
                            out[:5][np.isin(out[:5], [-1, 1])] = np.inf
                            out[5:][np.isin(out[5:], [-2, 0, 2])] = np.inf
                        
                        sort_idx = np.argsort(maxbands[:, 1])
                        maxbands_sorted = maxbands[sort_idx]
                        out_out=out[sort_idx]
                        sidebands_out=sidebands[np.argsort(sidebands[:,1])]

                        # Adding Frequnecy Difference features if requensted
                        if freq_diff_features:

                            freq = maxbands_sorted[:, 1]
                            diffs = freq[:, None] - freq[None, :]  

                            maxbands_sorted = np.hstack((maxbands_sorted, diffs))
                            maxbands_out=maxbands_sorted

                        else:
                            maxbands_out=maxbands_sorted


                        

                        all_sidebands.append(sidebands_out)
                        all_maxbands.append(maxbands_out)
                        all_outputs.append(out_out)
                        ang_pos.append([angle,position,hist_name])
                finally:
                    file1.Close()
                    file2.Close()
    all_sidebands=np.array(all_sidebands)
    all_maxbands=np.array(all_maxbands)
    all_outputs=np.array(all_outputs)
    ang_pos=np.array(ang_pos)

    return(all_sidebands,all_maxbands,all_outputs,ang_pos)

In [5]:
def process_jason(index, angle, position):
    jsn_name='LocustEventProperties.json'
    Jsn_info = index[angle][position][jsn_name]
    filepath = Jsn_info["filepath"]

    with open(filepath, "r") as f:
        jsn = json.load(f)

    return[jsn]

In [ ]:
def process_hist(index, angle, position, hist_name,rebin=None,noise=False,noise_spread=1):
    # This function is only called by other functions and is used to extract hist infomration from the indexing JASON file

    #load file
    hist_info = index[angle][position][hist_name]
    filepath = hist_info["filepath"]
    file = ROOT.TFile.Open(filepath,'READ')
    hist = file.Get(hist_name)

    #check file type
    if not hist:
        print("Missing:", hist_name)
        file.Close()
        return

    #clone it so i dont mess it up
    clone_name = f"{hist_name}_clone_{uuid.uuid4().hex}"
    hist_copy = hist.Clone(clone_name)
    hist_copy.SetDirectory(0)
    file.Close()

    if rebin is not None and rebin > 1:
        if not isinstance(rebin, int):
            raise ValueError("rebin must be an integer")
        old_level = ROOT.gErrorIgnoreLevel
        ROOT.gErrorIgnoreLevel = ROOT.kError
        hist_copy.Rebin(rebin)
        ROOT.gErrorIgnoreLevel = old_level

    nbins = hist_copy.GetNbinsX()
    counts = np.array([
            hist_copy.GetBinContent(i)
            for i in range(1, nbins + 1)
        ])
    centers=np.array([
            hist_copy.GetBinCenter(i)
            for i in range(1, nbins + 1)
        ])
    if noise:
        avg_power=np.mean(counts)
        noise = np.random.normal(0, avg_power*noise_spread, nbins) + 1j*np.random.normal(0, avg_power*noise_spread, nbins)
        noise_power = np.abs(noise)**2
        counts=counts+noise_power

    return(hist_copy, counts, centers)

In [ ]:
def print_hist(index, angle, position, hist_name):
    #prints the given hist
    hist, counts, centers=process_hist(index, angle, position, hist_name)
    del counts
    del centers
    for i in range(1, hist.GetNbinsX() + 1):
        print(hist.GetBinCenter(i), hist.GetBinContent(i))
    nbins = hist.GetNbinsX()
    print(nbins)
    hist.Delete()

In [2]:
def plot_hist(index, angle, position, hist_name, rebin=None, ax=None, max=None,noise=False,noise_spread=1,):
    #Plots the given hist

    hist, counts, centers=process_hist(index, angle, position, hist_name,rebin=rebin,noise=noise,noise_spread=noise_spread)

    nbins = hist.GetNbinsX()

    x =centers
    y =counts
    width =[]

    for i in range(1, nbins + 1):
        width.append(hist.GetBinWidth(i))
    x = np.array(x)
    y = np.array(y)
    width = np.array(width)

    if ax is None:
        fig, ax = plt.subplots()
    else:
        fig = ax.figure

    if max is None:
        bars=ax.bar(x, y, width=width, align="center")
    else:
        lims = (x >= max[0]) & (x <= max[1])
        bars=ax.bar(x[lims], y[lims], width=width[lims], align="center")

    ax.set_xlabel("Bin center")
    ax.set_ylabel("Counts")
    ax.set_title(f"{angle}, {position}, {hist_name}")

    return fig, ax, bars

In [3]:
def max_hist(index, angle, position, hist_name, gtr=None,btwn=None):
        #This prints the maximum value or values in a hist
        hist, counts, centers=process_hist(index, angle, position, hist_name)
        hist.Delete()
        maxes = []
        if btwn is not None:
            lims = (centers >= btwn[0]) & (centers <= btwn[1])
        else:
            lims = np.ones_like(counts, dtype=bool)
        counts= counts[lims]
        centers= centers[lims]  
        if gtr==None:
            mx=np.max(counts)
            i=int(np.where(counts==mx)[0][0])
            center=centers[i]
            power=counts[i]
            maxes.append([power,center])
        else:
            
            i=np.where(counts>gtr)[0]
            for a in range(len(i)):
                center=centers[int(i[a])]
                power=counts[int(i[a])]
                maxes.append([power,center])
        del counts
        del centers

        maxes=np.array(maxes)
        return(maxes)


In [4]:
def Time_density_hist(index, angle, position, rebin=None,freq_range=None):
    #Produces a time-density plot for a given angle and position
    def get_time_from_key(key):
        nums = re.findall(r"-?\d+\.?\d*", key)
        if len(nums) == 0:
            raise ValueError(f"No time/index number found in key: {key}")
        return float(nums[0])

    psd_keys = [
        key for key in index[angle][position]
        if "PSD" in key
    ]

    psd_keys = sorted(psd_keys, key=get_time_from_key)
    times = []
    rows = []
    freq_edges = None

    for key in psd_keys:
        time = get_time_from_key(key)

        hist,counts, centers = process_hist(index, angle, position, key, rebin)
        nbins= hist.GetNbinsX()
        edges = np.array(
            [hist.GetBinLowEdge(i) for i in range(1, nbins + 1)]
            + [hist.GetBinLowEdge(nbins) + hist.GetBinWidth(nbins)])
        del centers
        hist.Delete()
        times.append(time)
        rows.append(counts)
        if freq_edges is None:
            freq_edges = edges
    times = np.array(times)
    data = np.array(rows)

    if freq_range is not None:
        freq_centers = 0.5 * (freq_edges[:-1] + freq_edges[1:])
        lims = (freq_centers >= freq_range[0]) & (freq_centers <= freq_range[1])
        idx = np.where(lims)[0]

        first = idx[0]
        last = idx[-1]

        data = data[:, first:last + 1]
        freq_edges = freq_edges[first:last + 2]
        
    data = data.T

    # Make time bin edges from time centers
    if len(times) > 1:
        time_edges = np.zeros(len(times) + 1)
        time_edges[1:-1] = 0.5 * (times[:-1] + times[1:])
        time_edges[0] = times[0] - 0.5 * (times[1] - times[0])
        time_edges[-1] = times[-1] + 0.5 * (times[-1] - times[-2])
    else:
        time_edges = np.array([times[0] - 0.5, times[0] + 0.5])

    fig, ax = plt.subplots(figsize=(8, 6))

    mesh = ax.pcolormesh(
        time_edges,
        freq_edges,
        data,
        shading="auto"
    )

    fig.colorbar(mesh, ax=ax, label="Counts")

    ax.set_xlabel("Time")
    ax.set_ylabel("Frequency")
    ax.set_title(f"PSD density map: angle={angle}, position={position}")
    del data
    del rows
    del times


# stuff